# Step 6: Spark Structured Streaming

## Learning Objectives
1. Structured Streaming concepts and architecture
2. Source and Sink types
3. Micro-batch vs Continuous Processing
4. Output Modes: Append, Complete, Update
5. Watermark and Late Data handling
6. Window operations (Tumbling, Sliding, Session)
7. Streaming + batch data joins
8. Monitoring and checkpointing

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
import json
import os
import shutil

spark = SparkSession.builder \
    .appName("Step6-Structured-Streaming") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/data/warehouse") \
    .getOrCreate()

# Initialize working directories
BASE = "/home/jovyan/data/streaming"
subdirs = [
    "input/events", "input/watermark", "input/join", "input/filesink", "input/monitor",
    "output", "output/parquet",
    "checkpoint/events", "checkpoint/watermark", "checkpoint/join", "checkpoint/filesink", "checkpoint/monitor",
]
for d in subdirs:
    path = f"{BASE}/{d}"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

print(f"Spark version: {spark.version}")
print(f"✅ Spark UI: http://localhost:4040")
print(f"✅ Streaming data path: {BASE}")

Spark version: 3.5.0
✅ Spark UI: http://localhost:4040
✅ Streaming data path: /home/jovyan/data/streaming


---
## 1. Structured Streaming Concepts

Structured Streaming treats **continuously arriving data as an infinitely growing table**.

```
Time →  t1        t2        t3        t4
        ┌───┐     ┌───┐     ┌───┐     ┌───┐
new data│ A │     │ D │     │ G │     │ J │
        │ B │     │ E │     │ H │     │ K │
        │ C │     │ F │     │ I │     │ L │
        └─┬─┘     └─┬─┘     └─┬─┘     └─┬─┘
          ▼         ▼         ▼         ▼
┌─────────────────────────────────────────────┐
│       Unbounded Input Table                  │
│  A B C | D E F | G H I | J K L | ...        │
└──────────────────────┬──────────────────────┘
                       │ Query (DataFrame API / SQL)
                       ▼
┌─────────────────────────────────────────────┐
│          Result Table                        │
│  updated on every trigger                    │
└──────────────────────┬──────────────────────┘
                       │ Output Mode
                       ▼
                  External Sink
```

### Key Advantages
- Uses the **same DataFrame/SQL API** as batch
- Exactly-once guarantee (checkpoint-based)
- Event-time based processing (Watermark)

---
## 2. First Streaming Query: Rate Source

The `rate` source automatically generates a specified number of rows per second for testing.

In [2]:
# Rate source: generate 10 rows per second
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

print(f"Is streaming DataFrame? {rate_df.isStreaming}")
print(f"Schema:")
rate_df.printSchema()

Is streaming DataFrame? True
Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



In [3]:
# Simple transformation: assign category based on value
categorized = rate_df \
    .withColumn("category", 
        F.when(F.col("value") % 3 == 0, "A")
         .when(F.col("value") % 3 == 1, "B")
         .otherwise("C")
    ) \
    .withColumn("amount", (F.col("value") % 100) * 1.5)

# Output to memory sink (for testing/debugging)
query1 = categorized.writeStream \
    .format("memory") \
    .queryName("rate_test") \
    .outputMode("append") \
    .start()

print("Streaming started! Waiting 5 seconds...")
time.sleep(5)

# Check results from memory table
spark.sql("SELECT * FROM rate_test ORDER BY timestamp DESC LIMIT 10").show(truncate=False)

print(f"Total rows: {spark.sql('SELECT COUNT(*) FROM rate_test').collect()[0][0]}")

query1.stop()
print("\nQuery stopped")

Streaming started! Waiting 5 seconds...
+-----------------------+-----+--------+------+
|timestamp              |value|category|amount|
+-----------------------+-----+--------+------+
|2026-06-03 08:38:54.443|49   |B       |73.5  |
|2026-06-03 08:38:54.343|48   |A       |72.0  |
|2026-06-03 08:38:54.243|47   |C       |70.5  |
|2026-06-03 08:38:54.143|46   |B       |69.0  |
|2026-06-03 08:38:54.043|45   |A       |67.5  |
|2026-06-03 08:38:53.943|44   |C       |66.0  |
|2026-06-03 08:38:53.843|43   |B       |64.5  |
|2026-06-03 08:38:53.743|42   |A       |63.0  |
|2026-06-03 08:38:53.643|41   |C       |61.5  |
|2026-06-03 08:38:53.543|40   |B       |60.0  |
+-----------------------+-----+--------+------+

Total rows: 50

Query stopped


---
## 3. Output Modes

| Mode | Description | When to use |
|------|-------------|-------------|
| **Append** | Output only newly added rows | No aggregation, or aggregation with watermark |
| **Complete** | Output the entire result table | Aggregation queries only |
| **Update** | Output only changed rows | Most queries |

In [4]:
# Complete mode: output the full aggregation result every time
rate_df2 = spark.readStream.format("rate").option("rowsPerSecond", 20).load()

agg_df = rate_df2 \
    .withColumn("group", (F.col("value") % 5).cast("string")) \
    .groupBy("group") \
    .agg(
        F.count("*").alias("count"),
        F.avg("value").alias("avg_value")
    )

query2 = agg_df.writeStream \
    .format("memory") \
    .queryName("complete_test") \
    .outputMode("complete") \
    .start()

# Check 3 times at 3-second intervals
for i in range(3):
    time.sleep(3)
    print(f"\n=== Check {i+1} ({(i+1)*3}s elapsed) ===")
    spark.sql("SELECT * FROM complete_test ORDER BY group").show()

query2.stop()
print("💡 Complete mode: the full result is refreshed on every trigger.")
print("   Notice that count keeps increasing.")


=== Check 1 (3s elapsed) ===
+-----+-----+---------+
|group|count|avg_value|
+-----+-----+---------+
|    0|    8|     17.5|
|    1|    8|     18.5|
|    2|    8|     19.5|
|    3|    8|     20.5|
|    4|    8|     21.5|
+-----+-----+---------+


=== Check 2 (6s elapsed) ===
+-----+-----+---------+
|group|count|avg_value|
+-----+-----+---------+
|    0|   20|     47.5|
|    1|   20|     48.5|
|    2|   20|     49.5|
|    3|   20|     50.5|
|    4|   20|     51.5|
+-----+-----+---------+


=== Check 3 (9s elapsed) ===
+-----+-----+---------+
|group|count|avg_value|
+-----+-----+---------+
|    0|   36|     87.5|
|    1|   36|     88.5|
|    2|   36|     89.5|
|    3|   36|     90.5|
|    4|   36|     91.5|
+-----+-----+---------+

💡 Complete mode: the full result is refreshed on every trigger.
   Notice that count keeps increasing.


---
## 4. Streaming with File Source

The most common production pattern: **automatically process new files as they are added to a directory**

In [5]:
# Define JSON schema
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("event_type", StringType()),
    StructField("amount", DoubleType()),
    StructField("event_time", TimestampType()),
])

# Read streaming from file source
file_stream = spark.readStream \
    .format("json") \
    .schema(event_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(f"{BASE}/input/events/")

print(f"Streaming? {file_stream.isStreaming}")
file_stream.printSchema()

Streaming? True
root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [6]:
# Streaming aggregation query by event type
event_agg = file_stream \
    .groupBy("event_type") \
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount")
    )

query3 = event_agg.writeStream \
    .format("memory") \
    .queryName("event_summary") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{BASE}/checkpoint/events/") \
    .start()

print("Streaming query waiting... generating files.")

Streaming query waiting... generating files.


In [7]:
import random
from datetime import datetime, timedelta

def generate_events(batch_num, count=100):
    """Generate event JSON files and save to the input directory"""
    events = []
    base_time = datetime(2025, 6, 1, 10, 0, 0) + timedelta(minutes=batch_num * 5)
    
    for i in range(count):
        event = {
            "event_id": f"evt_{batch_num}_{i}",
            "user_id": random.randint(1, 1000),
            "event_type": random.choice(["purchase", "view", "click", "signup"]),
            "amount": round(random.uniform(0, 500), 2),
            "event_time": (base_time + timedelta(seconds=random.randint(0, 300))).isoformat()
        }
        events.append(json.dumps(event))
    
    path = f"{BASE}/input/events/batch_{batch_num}.json"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("\n".join(events))
    print(f"  Batch {batch_num}: {count} rows generated → {path}")

# Generate 3 batches sequentially and check results
for batch in range(3):
    generate_events(batch, count=200)
    time.sleep(5)  # wait for trigger
    
    print(f"\n  === Results after batch {batch} ===")
    spark.sql("SELECT * FROM event_summary ORDER BY event_count DESC").show()

query3.stop()
print("\n💡 New files are automatically processed as they are added to the directory.")
print("   Use maxFilesPerTrigger to limit how many files are processed per batch.")

  Batch 0: 200 rows generated → /home/jovyan/data/streaming/input/events/batch_0.json

  === Results after batch 0 ===
+----------+-----------+------------------+------------------+
|event_type|event_count|      total_amount|        avg_amount|
+----------+-----------+------------------+------------------+
|     click|         58|          15025.34|259.05758620689653|
|      view|         53|          11433.26|215.72188679245284|
|  purchase|         47|12913.509999999998| 274.7555319148936|
|    signup|         42| 9856.449999999997|234.67738095238087|
+----------+-----------+------------------+------------------+

  Batch 1: 200 rows generated → /home/jovyan/data/streaming/input/events/batch_1.json

  === Results after batch 1 ===
+----------+-----------+------------------+------------------+
|event_type|event_count|      total_amount|        avg_amount|
+----------+-----------+------------------+------------------+
|      view|        109|24890.980000000003|228.35761467889913|
|    

---
## 5. Watermark & Event Time

In the real world, events can **arrive out of order or late**.

```
Actual event time:  10:00  10:01  10:02  10:03  10:04
Arrival time:       10:00  10:02  10:01  10:05  10:03
                                  ↑ late!
                                          ↑ even later!
```

**Watermark** = a threshold that says "we will no longer wait for data older than this time"

```
Watermark = max event_time - allowed lateness

Example: watermark = 10 minutes
    Current max event_time = 10:30
    → Data before 10:20 is discarded (or removed from state)
    → Data between 10:20 ~ 10:30 is still accepted
```

In [8]:
# Window aggregation with Watermark

# Reset checkpoints
for d in ["input/watermark", "checkpoint/watermark"]:
    path = f"{BASE}/{d}"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

wm_stream = spark.readStream \
    .format("json") \
    .schema(event_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(f"{BASE}/input/watermark/")

# 10-minute watermark + 5-minute window aggregation
windowed = wm_stream \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        F.window("event_time", "5 minutes"),  # 5-minute Tumbling Window
        "event_type"
    ) \
    .agg(
        F.count("*").alias("count"),
        F.sum("amount").alias("total_amount")
    ) \
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_type",
        "count",
        "total_amount"
    )

query4 = windowed.writeStream \
    .format("memory") \
    .queryName("windowed_events") \
    .outputMode("update") \
    .option("checkpointLocation", f"{BASE}/checkpoint/watermark/") \
    .start()

print("Watermark streaming started!")

Watermark streaming started!


In [9]:
# Generate events with timestamps (some intentionally late)
def generate_timed_events(batch_num, base_hour, base_min, count=50, late_events=0, late_minutes=15):
    events = []
    base_time = datetime(2025, 6, 1, base_hour, base_min, 0)
    
    # Normal events
    for i in range(count):
        event = {
            "event_id": f"wm_{batch_num}_{i}",
            "user_id": random.randint(1, 100),
            "event_type": random.choice(["purchase", "view", "click"]),
            "amount": round(random.uniform(10, 200), 2),
            "event_time": (base_time + timedelta(seconds=random.randint(0, 290))).isoformat()
        }
        events.append(json.dumps(event))
    
    # Late-arriving events (past timestamps)
    for i in range(late_events):
        late_time = base_time - timedelta(minutes=late_minutes)
        event = {
            "event_id": f"wm_late_{batch_num}_{i}",
            "user_id": random.randint(1, 100),
            "event_type": "purchase",
            "amount": round(random.uniform(100, 500), 2),
            "event_time": (late_time + timedelta(seconds=random.randint(0, 290))).isoformat()
        }
        events.append(json.dumps(event))
    
    path = f"{BASE}/input/watermark/batch_{batch_num}.json"
    with open(path, "w") as f:
        f.write("\n".join(events))
    
    late_info = f" + {late_events} late events (-{late_minutes}min)" if late_events else ""
    print(f"Batch {batch_num}: {count} rows based at {base_hour}:{base_min:02d}{late_info}")

# Batch 0: 10:00 ~ 10:05
generate_timed_events(0, 10, 0, count=50)
time.sleep(5)
print("\n--- Window results ---")
spark.sql("SELECT * FROM windowed_events ORDER BY window_start, event_type").show(truncate=False)

# Batch 1: 10:05 ~ 10:10
generate_timed_events(1, 10, 5, count=50)
time.sleep(5)
print("\n--- Window results ---")
spark.sql("SELECT * FROM windowed_events ORDER BY window_start, event_type").show(truncate=False)

# Batch 2: based at 10:20 + late events for 10:05 (15 min late → exceeds 10-min watermark!)
generate_timed_events(2, 10, 20, count=50, late_events=10, late_minutes=15)
time.sleep(5)
print("\n--- Window results (with late events) ---")
spark.sql("SELECT * FROM windowed_events ORDER BY window_start, event_type").show(truncate=False)

print("""
💡 Watermark = 10 min, current max event_time ≈ 10:25
   → Watermark threshold = 10:15
   → 10:05 events are older than the watermark (10:15) → may be dropped
   → Notice that count for the 10:00~10:05 window no longer increases.
""")

query4.stop()

Batch 0: 50 rows based at 10:00

--- Window results ---
+-------------------+-------------------+----------+-----+------------------+
|window_start       |window_end         |event_type|count|total_amount      |
+-------------------+-------------------+----------+-----+------------------+
|2025-06-01 10:00:00|2025-06-01 10:05:00|click     |22   |2190.8            |
|2025-06-01 10:00:00|2025-06-01 10:05:00|purchase  |13   |1210.8200000000002|
|2025-06-01 10:00:00|2025-06-01 10:05:00|view      |15   |1187.86           |
+-------------------+-------------------+----------+-----+------------------+

Batch 1: 50 rows based at 10:05

--- Window results ---
+-------------------+-------------------+----------+-----+------------------+
|window_start       |window_end         |event_type|count|total_amount      |
+-------------------+-------------------+----------+-----+------------------+
|2025-06-01 10:00:00|2025-06-01 10:05:00|click     |22   |2190.8            |
|2025-06-01 10:00:00|2025-06-

---
## 6. Window Operation Types

```
Tumbling Window (fixed window)
  |--5min--|--5min--|--5min--|--5min--|
  no overlap, no gaps

Sliding Window
  |----10min----|
       |----10min----|
            |----10min----|
  window size > slide interval → overlap

Session Window — Spark 3.2+
  |--event--gap--| |--event--event--gap--| |--event--|
  a new session starts when the gap between events exceeds the threshold
```

In [10]:
# Compare window operations on batch data (easier to understand)
from datetime import datetime

sample_events = [
    ("u1", "purchase", 100.0, datetime(2025, 6, 1, 10, 1)),
    ("u2", "purchase", 200.0, datetime(2025, 6, 1, 10, 3)),
    ("u1", "purchase", 150.0, datetime(2025, 6, 1, 10, 6)),
    ("u3", "purchase", 300.0, datetime(2025, 6, 1, 10, 8)),
    ("u1", "purchase", 120.0, datetime(2025, 6, 1, 10, 12)),
    ("u2", "purchase", 250.0, datetime(2025, 6, 1, 10, 14)),
    ("u3", "purchase", 180.0, datetime(2025, 6, 1, 10, 18)),
    ("u1", "purchase", 90.0,  datetime(2025, 6, 1, 10, 22)),
]

events_df = spark.createDataFrame(sample_events, ["user_id", "event_type", "amount", "event_time"])
events_df.show(truncate=False)

+-------+----------+------+-------------------+
|user_id|event_type|amount|event_time         |
+-------+----------+------+-------------------+
|u1     |purchase  |100.0 |2025-06-01 10:01:00|
|u2     |purchase  |200.0 |2025-06-01 10:03:00|
|u1     |purchase  |150.0 |2025-06-01 10:06:00|
|u3     |purchase  |300.0 |2025-06-01 10:08:00|
|u1     |purchase  |120.0 |2025-06-01 10:12:00|
|u2     |purchase  |250.0 |2025-06-01 10:14:00|
|u3     |purchase  |180.0 |2025-06-01 10:18:00|
|u1     |purchase  |90.0  |2025-06-01 10:22:00|
+-------+----------+------+-------------------+



In [11]:
# Tumbling Window: fixed 10-minute window
print("=== Tumbling Window (10 min) ===")
events_df.groupBy(
    F.window("event_time", "10 minutes")
).agg(
    F.count("*").alias("count"),
    F.sum("amount").alias("total")
).select(
    F.col("window.start").alias("start"),
    F.col("window.end").alias("end"),
    "count", "total"
).orderBy("start").show(truncate=False)

# Sliding Window: 10-minute window, 5-minute slide
print("=== Sliding Window (10-min window, 5-min slide) ===")
events_df.groupBy(
    F.window("event_time", "10 minutes", "5 minutes")
).agg(
    F.count("*").alias("count"),
    F.sum("amount").alias("total")
).select(
    F.col("window.start").alias("start"),
    F.col("window.end").alias("end"),
    "count", "total"
).orderBy("start").show(truncate=False)

print("💡 In a Sliding Window, events can belong to multiple windows.")
print("   → Useful for moving averages, statistics over the last N minutes, etc.")

=== Tumbling Window (10 min) ===
+-------------------+-------------------+-----+-----+
|start              |end                |count|total|
+-------------------+-------------------+-----+-----+
|2025-06-01 10:00:00|2025-06-01 10:10:00|4    |750.0|
|2025-06-01 10:10:00|2025-06-01 10:20:00|3    |550.0|
|2025-06-01 10:20:00|2025-06-01 10:30:00|1    |90.0 |
+-------------------+-------------------+-----+-----+

=== Sliding Window (10-min window, 5-min slide) ===
+-------------------+-------------------+-----+-----+
|start              |end                |count|total|
+-------------------+-------------------+-----+-----+
|2025-06-01 09:55:00|2025-06-01 10:05:00|2    |300.0|
|2025-06-01 10:00:00|2025-06-01 10:10:00|4    |750.0|
|2025-06-01 10:05:00|2025-06-01 10:15:00|4    |820.0|
|2025-06-01 10:10:00|2025-06-01 10:20:00|3    |550.0|
|2025-06-01 10:15:00|2025-06-01 10:25:00|2    |270.0|
|2025-06-01 10:20:00|2025-06-01 10:30:00|1    |90.0 |
+-------------------+-------------------+-----+---

In [12]:
# Session Window: per-user session (5-minute gap)
print("=== Session Window (per user, 5-minute gap) ===")
events_df.groupBy(
    "user_id",
    F.session_window("event_time", "5 minutes")
).agg(
    F.count("*").alias("events_in_session"),
    F.sum("amount").alias("session_total"),
    F.min("event_time").alias("first_event"),
    F.max("event_time").alias("last_event")
).select(
    "user_id",
    F.col("session_window.start").alias("session_start"),
    F.col("session_window.end").alias("session_end"),
    "events_in_session", "session_total"
).orderBy("user_id", "session_start").show(truncate=False)

print("💡 Session Window is key for user behavior analysis:")
print("   → Revenue per session, session duration, bounce rate, etc.")

=== Session Window (per user, 5-minute gap) ===
+-------+-------------------+-------------------+-----------------+-------------+
|user_id|session_start      |session_end        |events_in_session|session_total|
+-------+-------------------+-------------------+-----------------+-------------+
|u1     |2025-06-01 10:01:00|2025-06-01 10:11:00|2                |250.0        |
|u1     |2025-06-01 10:12:00|2025-06-01 10:17:00|1                |120.0        |
|u1     |2025-06-01 10:22:00|2025-06-01 10:27:00|1                |90.0         |
|u2     |2025-06-01 10:03:00|2025-06-01 10:08:00|1                |200.0        |
|u2     |2025-06-01 10:14:00|2025-06-01 10:19:00|1                |250.0        |
|u3     |2025-06-01 10:08:00|2025-06-01 10:13:00|1                |300.0        |
|u3     |2025-06-01 10:18:00|2025-06-01 10:23:00|1                |180.0        |
+-------+-------------------+-------------------+-----------------+-------------+

💡 Session Window is key for user behavior analysi

---
## 7. Streaming + Batch Join (Stream-Static Join)

In [13]:
# Static table: product info
products = spark.createDataFrame([
    (1, "Laptop",   "Electronics", 1200000),
    (2, "Keyboard", "Electronics",  150000),
    (3, "Sneakers", "Sports",        89000),
    (4, "Novel",    "Books",         15000),
    (5, "Coat",     "Apparel",      250000),
], ["product_id", "product_name", "category", "price"])

# Reset checkpoints
for d in ["input/join", "checkpoint/join"]:
    path = f"{BASE}/{d}"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

# Order stream schema
order_schema = StructType([
    StructField("order_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("order_time", TimestampType()),
])

order_stream = spark.readStream \
    .format("json") \
    .schema(order_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(f"{BASE}/input/join/")

# Join streaming orders with static products
enriched = order_stream.join(products, "product_id") \
    .withColumn("total_price", F.col("price") * F.col("quantity")) \
    .select("order_id", "user_id", "product_name", "category", 
            "quantity", "price", "total_price", "order_time")

query5 = enriched.writeStream \
    .format("memory") \
    .queryName("enriched_orders") \
    .outputMode("append") \
    .option("checkpointLocation", f"{BASE}/checkpoint/join/") \
    .start()

print("Stream-Static Join started!")

Stream-Static Join started!


In [14]:
# Generate order data
def generate_orders(batch_num, count=20):
    orders = []
    base_time = datetime(2025, 6, 1, 14, 0, 0) + timedelta(minutes=batch_num * 10)
    
    for i in range(count):
        order = {
            "order_id": f"ord_{batch_num}_{i}",
            "user_id": random.randint(1, 100),
            "product_id": random.randint(1, 5),
            "quantity": random.randint(1, 3),
            "order_time": (base_time + timedelta(seconds=random.randint(0, 600))).isoformat()
        }
        orders.append(json.dumps(order))
    
    path = f"{BASE}/input/join/orders_{batch_num}.json"
    with open(path, "w") as f:
        f.write("\n".join(orders))
    print(f"  Order batch {batch_num}: {count} rows generated")

# Generate orders and check results
for batch in range(2):
    generate_orders(batch)
    time.sleep(5)
    
    print(f"\n  === Batch {batch} results ===")
    spark.sql("""
        SELECT product_name, category,
               COUNT(*) as orders, 
               SUM(total_price) as revenue
        FROM enriched_orders 
        GROUP BY product_name, category
        ORDER BY revenue DESC
    """).show()

query5.stop()

print("""
💡 Stream-Static Join:
   - Combines streaming data with static reference data
   - Production example: enriching real-time orders with product/customer info
   - The static table is re-read on every trigger to reflect the latest state
   - Supports Inner Join and Left Outer Join
""")

  Order batch 0: 20 rows generated

  === Batch 0 results ===
+------------+-----------+------+-------+
|product_name|   category|orders|revenue|
+------------+-----------+------+-------+
|      Laptop|Electronics|     5|8400000|
|        Coat|    Apparel|     6|3000000|
|    Sneakers|     Sports|     2| 356000|
|    Keyboard|Electronics|     1| 300000|
|       Novel|      Books|     6| 195000|
+------------+-----------+------+-------+

  Order batch 1: 20 rows generated

  === Batch 1 results ===
+------------+-----------+------+--------+
|product_name|   category|orders| revenue|
+------------+-----------+------+--------+
|      Laptop|Electronics|    11|22800000|
|        Coat|    Apparel|     8| 4000000|
|    Keyboard|Electronics|     5| 1200000|
|    Sneakers|     Sports|     5|  623000|
|       Novel|      Books|    11|  270000|
+------------+-----------+------+--------+


💡 Stream-Static Join:
   - Combines streaming data with static reference data
   - Production example: enric

---
## 8. Output to File Sink (Production Pattern)

In [15]:
# Initialize checkpoint
for d in ["input/filesink", "output/parquet", "checkpoint/filesink"]:
    path = f"{BASE}/{d}"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

filesink_stream = spark.readStream \
    .format("json") \
    .schema(event_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(f"{BASE}/input/filesink/")

# Output to Parquet files (with partitioning)
query6 = filesink_stream \
    .withColumn("event_date", F.to_date("event_time")) \
    .withColumn("event_hour", F.hour("event_time")) \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", f"{BASE}/output/parquet/") \
    .option("checkpointLocation", f"{BASE}/checkpoint/filesink/") \
    .partitionBy("event_date", "event_hour") \
    .trigger(processingTime="5 seconds") \
    .start()

# Generate data
for batch in range(3):
    generate_events(batch + 10, count=100)
    # write input path to file sink
    src = f"{BASE}/input/events/batch_{batch + 10}.json"
    dst = f"{BASE}/input/filesink/batch_{batch + 10}.json"
    shutil.copy2(src, dst)
    time.sleep(6)

query6.stop()

# Check output files
print("\n=== Output Parquet file structure ===")
for root, dirs, files in os.walk(f"{BASE}/output/parquet/"):
    level = root.replace(f"{BASE}/output/parquet/", "").count(os.sep)
    indent = "  " * level
    dirname = os.path.basename(root)
    if dirname and not dirname.startswith("_"):
        print(f"{indent}{dirname}/")
    parquets = [f for f in files if f.endswith(".parquet")]
    if parquets:
        print(f"{indent}  {len(parquets)} parquet files")

# Verify by reading back
print("\n=== Reading from Parquet ===")
spark.read.parquet(f"{BASE}/output/parquet/").show(5)

  Batch 10: 100 rows generated → /home/jovyan/data/streaming/input/events/batch_10.json
  Batch 11: 100 rows generated → /home/jovyan/data/streaming/input/events/batch_11.json
  Batch 12: 100 rows generated → /home/jovyan/data/streaming/input/events/batch_12.json

=== Output Parquet file structure ===
event_date=2025-06-01/
  event_hour=11/
    1 parquet files
  event_hour=10/
    2 parquet files

=== Reading from Parquet ===
+--------+-------+----------+------+-------------------+----------+----------+
|event_id|user_id|event_type|amount|         event_time|event_date|event_hour|
+--------+-------+----------+------+-------------------+----------+----------+
|evt_11_0|    487|  purchase|449.36|2025-06-01 10:55:54|2025-06-01|        10|
|evt_11_1|    230|  purchase|320.82|2025-06-01 10:55:19|2025-06-01|        10|
|evt_11_2|    938|    signup|102.36|2025-06-01 10:55:05|2025-06-01|        10|
|evt_11_3|     78|  purchase|439.87|2025-06-01 10:57:19|2025-06-01|        10|
|evt_11_4|    272

---
## 9. Query Monitoring

In [16]:
# Demo: query status monitoring

mon_stream = spark.readStream \
    .format('rate') \
    .option('rowsPerSecond', 50) \
    .load()

mon_query = mon_stream \
    .withColumn('group', (F.col('value') % 10).cast('string')) \
    .groupBy('group').count() \
    .writeStream \
    .format('memory') \
    .queryName('monitor_test') \
    .outputMode('complete') \
    .start()

time.sleep(5)

# Check query status
status = mon_query.status
progress = mon_query.lastProgress

print('=== Query Status ===')
for k, v in status.items():
    print(f'  {k}: {v}')

if progress:
    print()
    print('=== Last Progress ===')
    batch_id = progress.get('batchId', 'N/A')
    input_rps = progress.get('inputRowsPerSecond', 'N/A')
    proc_rps = progress.get('processedRowsPerSecond', 'N/A')
    print(f'  Batch ID:             {batch_id}')
    print(f'  Input rows/sec:       {input_rps}')
    print(f'  Processed rows/sec:   {proc_rps}')

    durations = progress.get('durationMs', {})
    if durations:
        print('  Processing time:')
        for phase, ms in durations.items():
            print(f'    {phase}: {ms}ms')

    for op in progress.get('stateOperators', []):
        rows = op.get('numRowsTotal', 'N/A')
        mem = op.get('memoryUsedBytes', 'N/A')
        print(f'  State row count: {rows}')
        print(f'  State size:      {mem} bytes')

mon_query.stop()

print('''
🔍 Streaming monitoring checkpoints:

  1. inputRowsPerSecond vs processedRowsPerSecond
     → If processing speed is slower than input, lag accumulates

  2. durationMs
     → Time spent in each phase (addBatch, getOffset, etc.)
     → If longer than the trigger interval, backlog builds up

  3. stateOperators (for aggregations/joins)
     → Continuously growing state size signals a memory problem
     → Must use Watermark to clean up stale state

  4. Spark UI > Structured Streaming tab
     → Processing time graph per batch
     → Input/processed rows per second trends
''')


=== Query Status ===
  message: Waiting for data to arrive
  isDataAvailable: False
  isTriggerActive: False

=== Last Progress ===
  Batch ID:             4
  Input rows/sec:       4545.454545454546
  Processed rows/sec:   490.1960784313726
  Processing time:
    addBatch: 74ms
    commitOffsets: 7ms
    getBatch: 0ms
    latestOffset: 0ms
    queryPlanning: 6ms
    triggerExecution: 102ms
    walCommit: 14ms
  State row count: 10
  State size:      4880 bytes

🔍 Streaming monitoring checkpoints:

  1. inputRowsPerSecond vs processedRowsPerSecond
     → If processing speed is slower than input, lag accumulates

  2. durationMs
     → Time spent in each phase (addBatch, getOffset, etc.)
     → If longer than the trigger interval, backlog builds up

  3. stateOperators (for aggregations/joins)
     → Continuously growing state size signals a memory problem
     → Must use Watermark to clean up stale state

  4. Spark UI > Structured Streaming tab
     → Processing time graph per batch
 

---
## 10. Trigger Modes

In [17]:
print("""
=== Trigger Modes ===

1. Default (micro-batch)
   .trigger()  # next batch starts immediately after the previous one completes

2. Fixed Interval
   .trigger(processingTime="10 seconds")
   → runs one batch every 10 seconds
   → if a batch takes longer than 10s, the next one starts immediately

3. Once (single run)
   .trigger(once=True)  # deprecated in 3.4
   → processes all available data in one shot then stops
   → useful when combined with a scheduler (Airflow, etc.)

4. Available Now (Spark 3.3+)
   .trigger(availableNow=True)
   → similar to once, but splits processing across multiple micro-batches
   → more memory-efficient

Production recommendations:
  - Real-time dashboards: processingTime="5 seconds"
  - Near-real-time ETL:   processingTime="1 minute"
  - Batch-style streaming: availableNow=True + scheduler
""")


=== Trigger Modes ===

1. Default (micro-batch)
   .trigger()  # next batch starts immediately after the previous one completes

2. Fixed Interval
   .trigger(processingTime="10 seconds")
   → runs one batch every 10 seconds
   → if a batch takes longer than 10s, the next one starts immediately

3. Once (single run)
   .trigger(once=True)  # deprecated in 3.4
   → processes all available data in one shot then stops
   → useful when combined with a scheduler (Airflow, etc.)

4. Available Now (Spark 3.3+)
   .trigger(availableNow=True)
   → similar to once, but splits processing across multiple micro-batches
   → more memory-efficient

Production recommendations:
  - Real-time dashboards: processingTime="5 seconds"
  - Near-real-time ETL:   processingTime="1 minute"
  - Batch-style streaming: availableNow=True + scheduler



---
## 📝 Key Takeaways

| Concept | Description |
|---------|-------------|
| **Structured Streaming** | Processes streams as an unbounded table; same API as batch |
| **Source** | rate (testing), file (JSON/Parquet/CSV), Kafka |
| **Sink** | memory (testing), file, console, Kafka, foreach |
| **Output Mode** | Append (new rows), Complete (full result), Update (changed rows) |
| **Watermark** | Defines the allowed lateness for late-arriving data |
| **Tumbling Window** | Fixed-size, non-overlapping time windows |
| **Sliding Window** | Overlapping time windows (for moving averages, etc.) |
| **Session Window** | Dynamic windows based on inter-event gap |
| **Stream-Static Join** | Join streaming data with a static batch table |
| **Checkpoint** | State persistence for fault recovery (exactly-once) |
| **Trigger** | Batch execution interval (default, fixed, once, availableNow) |

### Production Checklist
1. ✅ Always set a Watermark on aggregation queries (prevents unbounded state growth)
2. ✅ Use reliable storage for checkpoint paths (HDFS, S3)
3. ✅ Scale up if inputRowsPerSecond > processedRowsPerSecond
4. ✅ Be careful about checkpoint compatibility when changing schemas
5. ✅ Kafka source is the production standard (covered in the next step)

### Next Step (Step 7)
- Building a production data pipeline
- ETL pipeline design
- Using Delta Lake
- Partitioning strategies and data lake patterns

In [18]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
